# Ginzburg–Landau — SLD

**Protocol:** unseen initial conditions.

This is the supplied representative split with held-out IC 7, seven training trajectories and two validation trajectories. Python `random.sample` is unseeded in the original; its seed is intentionally preserved as supplied. The paper aggregates ten held-out-trajectory folds.

Run the configuration cell followed by the experiment cell. Generate the MATLAB datasets with `data_generation/Ginzburg-Landau/Ginzburg-Landau_data.ipynb` first. The experiment cell defines the model, loads the data and starts the original 1,000-epoch training run. It writes the best validation checkpoint and training history to the printed results directory. A test loader is constructed, but the supplied code does not run test evaluation.

Dependencies: PyTorch, NumPy, SciPy and Matplotlib. The scientific implementation and hyperparameters are preserved; notebook continuations were merged to restore valid execution. Original file: `2D-GL-CaseB.ipynb`.


In [ ]:
from pathlib import Path
import os

# Start Jupyter in the repository or any directory beneath it.
def _find_repository_root():
    working_dir = Path.cwd().resolve()
    for candidate in (working_dir, *working_dir.parents):
        if (candidate / "README.md").is_file() and (candidate / "data_generation").is_dir():
            return candidate
    raise RuntimeError("Cannot find the SLD repository. Start Jupyter from the repository directory.")

ROOT = _find_repository_root()
DATA_ROOT = Path(os.environ.get("SLD_DATA_ROOT", str(ROOT / "data"))).expanduser()
RESULTS_ROOT = Path(os.environ.get("SLD_RESULTS_ROOT", str(ROOT / "results"))).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = ROOT / DATA_ROOT
if not RESULTS_ROOT.is_absolute():
    RESULTS_ROOT = ROOT / RESULTS_ROOT
DATA_DIR = DATA_ROOT / "ginzburg_landau"
RESULTS_DIR = RESULTS_ROOT / "task1_unseen_ic" / "ginzburg_landau"
print(f"Repository: {ROOT}")
print(f"Data: {DATA_DIR}")
print(f"Results: {RESULTS_DIR}")


In [ ]:
""" SLD for the 2D complex Ginzburg-Landau equation. """
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.optim import lr_scheduler
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as scio
import time
import os
import math
import random
from torch.nn.utils import weight_norm, spectral_norm


torch.manual_seed(42)
np.random.seed(42)
torch.set_default_dtype(torch.float32)


# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directory structure
base_dir = ROOT
data_dir = DATA_DIR
model_dir = RESULTS_DIR / 'model'
fig_dir = RESULTS_DIR / 'figures'


# Create directories if they don't exist
model_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir.absolute()}")
print(f"Model directory: {model_dir.absolute()}")
print(f"Figure directory: {fig_dir.absolute()}")


lapl_op = [[[[    0,   0, -1/12,   0,     0],
             [    0,   0,   4/3,   0,     0],
             [-1/12, 4/3,    -5, 4/3, -1/12],
             [    0,   0,   4/3,   0,     0],
             [    0,   0, -1/12,   0,     0]]]]


def initialize_weights(module):

    c = 1
    if isinstance(module, nn.Conv2d):
        module.weight.data.uniform_(-c*np.sqrt(1 / (3 * 3 * 320)), 
            c*np.sqrt(1 / (3 * 3 * 320)))

    if isinstance(module, nn.Conv1d):
        module.weight.data.uniform_(-c*np.sqrt(1 / (3 * 3 * 320)), 
            c*np.sqrt(1 / (3 * 3 * 320)))
     
    elif isinstance(module, nn.Linear):
        module.bias.data.zero_()

class ShiftMean(nn.Module):
    """
    支持 x 形状:
      - [T,B,C,H,W]
      - [B,C,H,W]
    """
    def __init__(self, mean, std):
        super().__init__()
        mean = torch.as_tensor(mean).float().view(1, 1, -1, 1, 1)  # [1,1,C,1,1]
        std  = torch.as_tensor(std ).float().view(1, 1, -1, 1, 1)
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)

    def forward(self, x, mode: str):
        if x.dim() == 4:
            # [B,C,H,W] -> [1,B,C,H,W] 复用同一套广播
            x_in = x.unsqueeze(0)
            squeeze_back = True
        else:
            x_in = x
            squeeze_back = False

        if mode == "sub":
            y = (x_in - self.mean) / (self.std + 1e-12)
        elif mode == "add":
            y = x_in * (self.std + 1e-12) + self.mean
        else:
            raise NotImplementedError

        return y.squeeze(0) if squeeze_back else y
        
def haar_iwt2x(ll, lh, hl, hh):
    """
    Inverse Haar wavelet synthesis (2x upsampling).
    ll/lh/hl/hh: [B, C, H, W] -> out: [B, C, 2H, 2W]
    """
    x00 = (ll + lh + hl + hh) * 0.5
    x01 = (ll - lh + hl - hh) * 0.5
    x10 = (ll + lh - hl - hh) * 0.5
    x11 = (ll - lh - hl + hh) * 0.5

    B, C, H, W = ll.shape
    out = torch.empty(B, C, 2 * H, 2 * W, device=ll.device, dtype=ll.dtype)
    out[:, :, 0::2, 0::2] = x00
    out[:, :, 0::2, 1::2] = x01
    out[:, :, 1::2, 0::2] = x10
    out[:, :, 1::2, 1::2] = x11
    return out

class PairwiseLearnedInterpolator(nn.Module):
    """
    对一对相邻 LR 帧 (x0, x1) 生成中间细分帧。
    输入:
        x0, x1: [B, C, H, W]
        alpha:  [B, 1, H, W] or [1,1,1,1]
    输出:
        y:      [B, C, H, W]
    """
    def __init__(self, channels, hidden=32):
        super().__init__()
        in_ch = channels * 2 + 1  # x0, x1, alpha map

        self.fuse = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1, padding_mode='circular'),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, padding_mode='circular'),
            nn.GELU()
        )

        self.w_head = nn.Conv2d(hidden, 2 * channels, 3, padding=1, padding_mode='circular')
        self.r_head = nn.Conv2d(hidden, channels, 3, padding=1, padding_mode='circular')

    def forward(self, x0, x1, alpha):
        B, C, H, W = x0.shape

        if alpha.dim() == 0:
            alpha = alpha.view(1, 1, 1, 1).to(x0)
        if alpha.shape[0] == 1 and B > 1:
            alpha = alpha.expand(B, 1, H, W)
        elif alpha.shape[-2:] == (1, 1):
            alpha = alpha.expand(B, 1, H, W)

        base = (1.0 - alpha) * x0 + alpha * x1

        feat = torch.cat([x0, x1, alpha], dim=1)
        feat = self.fuse(feat)

        w = self.w_head(feat)                      # [B, 2C, H, W]
        r = self.r_head(feat)                     # [B, C, H, W]

        w0, w1 = torch.chunk(w, 2, dim=1)
        w0 = torch.sigmoid(w0)
        w1 = torch.sigmoid(w1)

        out = w0 * x0 + w1 * x1 + r
        out = out + base
        return out
        
class HaarPyramidSynthesis(nn.Module):

    def __init__(self, out_channels: int, scale: int):
        super().__init__()
        assert scale >= 2 and (scale & (scale - 1) == 0), "scale must be power of 2"
        self.C = out_channels
        self.scale = scale
        self.L = int(math.log2(scale))

    def forward(self, packed: torch.Tensor) -> torch.Tensor:
        B, ch, H, W = packed.shape
        C = self.C
        assert ch == C * (self.scale ** 2), f"expected {C*(self.scale**2)} channels, got {ch}"

        idx = 0
        # coarsest (at base H,W): 4*C
        ll = packed[:, idx:idx + C]; idx += C
        lh = packed[:, idx:idx + C]; idx += C
        hl = packed[:, idx:idx + C]; idx += C
        hh = packed[:, idx:idx + C]; idx += C

        x = haar_iwt2x(ll, lh, hl, hh)  # [B,C,2H,2W]

        # finer levels
        for m in range(1, self.L):
            r = 2 ** m
            per = C * (r * r)

            lh_p = packed[:, idx:idx + per]; idx += per
            hl_p = packed[:, idx:idx + per]; idx += per
            hh_p = packed[:, idx:idx + per]; idx += per

            # unpack to current spatial size (H*r, W*r)
            lh = F.pixel_shuffle(lh_p, r)
            hl = F.pixel_shuffle(hl_p, r)
            hh = F.pixel_shuffle(hh_p, r)

            x = haar_iwt2x(x, lh, hl, hh)

        assert idx == ch
        return x


class WaveletSRHead(nn.Module):
    def __init__(self, in_ch, out_ch, scale, conv3x3_factory):
        super().__init__()
        self.proj = conv3x3_factory(in_ch, out_ch * (scale ** 2))
        self.synth = HaarPyramidSynthesis(out_channels=out_ch, scale=scale)

    def forward(self, feat):
        packed = self.proj(feat)
        return self.synth(packed)


class ConvNeXt_GRU_Cell(nn.Module):
    """
    ConvNeXt-style GRU cell used as a continuous-time GRU derivative:
        dh/dt = (1 - z) * (h_tilde - h) + alpha * Lap(h) + beta_x * Dx(h) + beta_y * Dy(h)

    Notes:
        - Dx, Dy, Lap are fixed depthwise conv operators
        - alpha, beta_x, beta_y are learnable per-channel parameters
    """
    def __init__(self, dim, custom_act=None):
        super().__init__()
        self.dim = dim

        # Gates: r and z together
        self.gate_conv = nn.Sequential(
            nn.Conv2d(dim * 2, dim * 2, kernel_size=7, padding=3, groups=dim * 2, padding_mode='circular'),
            nn.GroupNorm(1, dim * 2),
            nn.Conv2d(dim * 2, dim * 2, kernel_size=1),
            nn.Sigmoid()
        )

        # Candidate path (ConvNeXt-ish bottleneck)
        self.candidate_dw = nn.Conv2d(dim * 2, dim * 2, kernel_size=7, padding=3, groups=dim * 2, padding_mode='circular')
        self.candidate_norm = nn.GroupNorm(1, dim * 2)
        self.candidate_pw1 = nn.Conv2d(dim * 2, dim * 4, kernel_size=1)
        self.act = custom_act if custom_act is not None else nn.GELU()
        self.candidate_pw2 = nn.Conv2d(dim * 4, dim, kernel_size=1)

        # ------------------------------------------------------------------
        # Fixed differential operators: depthwise Dx, Dy, Lap
        # ------------------------------------------------------------------
        self.dx = nn.Conv2d(
            dim, dim, kernel_size=3, padding=1, groups=dim,
            bias=False, padding_mode='circular'
        )
        self.dy = nn.Conv2d(
            dim, dim, kernel_size=3, padding=1, groups=dim,
            bias=False, padding_mode='circular'
        )
        self.lap = nn.Conv2d(
            dim, dim, kernel_size=3, padding=1, groups=dim,
            bias=False, padding_mode='circular'
        )

        self._init_fixed_kernels()

        # freeze fixed operators
        for p in self.dx.parameters():
            p.requires_grad = False
        for p in self.dy.parameters():
            p.requires_grad = False
        for p in self.lap.parameters():
            p.requires_grad = False

        # ------------------------------------------------------------------
        # Learnable coefficients (per-channel)
        # shape: [1, C, 1, 1] for broadcasting
        # small init to keep this branch weak at start
        # ------------------------------------------------------------------
        self.alpha  = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.beta_x = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.beta_y = nn.Parameter(torch.zeros(1, dim, 1, 1))

    def _init_fixed_kernels(self):
        """
        Central-difference-like first derivatives and 5-point Laplacian.
        """
        device = self.dx.weight.device
        dtype = self.dx.weight.dtype

        dx_kernel = torch.tensor(
            [[0.0, 0.0, 0.0],
             [-0.5, 0.0, 0.5],
             [0.0, 0.0, 0.0]],
            dtype=dtype, device=device
        )

        dy_kernel = torch.tensor(
            [[0.0, -0.5, 0.0],
             [0.0,  0.0, 0.0],
             [0.0,  0.5, 0.0]],
            dtype=dtype, device=device
        )

        lap_kernel = torch.tensor(
            [[0.0,  1.0, 0.0],
             [1.0, -4.0, 1.0],
             [0.0,  1.0, 0.0]],
            dtype=dtype, device=device
        )

        dx_weight = torch.zeros_like(self.dx.weight.data)   # [C,1,3,3]
        dy_weight = torch.zeros_like(self.dy.weight.data)
        lap_weight = torch.zeros_like(self.lap.weight.data)

        dx_weight[:, 0, :, :] = dx_kernel
        dy_weight[:, 0, :, :] = dy_kernel
        lap_weight[:, 0, :, :] = lap_kernel

        with torch.no_grad():
            self.dx.weight.copy_(dx_weight)
            self.dy.weight.copy_(dy_weight)
            self.lap.weight.copy_(lap_weight)

    def forward(self, h, x_input):
        """
        h: [B, dim, H, W]
        x_input: [B, dim, H, W]
        """
        combined = torch.cat([h, x_input], dim=1)  # [B, 2*dim, H, W]

        gates = self.gate_conv(combined)
        r, z = torch.split(gates, self.dim, dim=1)

        h_reset = r * h
        combined_reset = torch.cat([h_reset, x_input], dim=1)

        out = self.candidate_dw(combined_reset)
        out = self.candidate_norm(out)
        out = self.candidate_pw1(out)
        out = self.act(out)
        out = self.candidate_pw2(out)  # h_tilde

        # Original ODE-GRU term
        dh_dt_main = (1 - z) * (out - h)

        # Fixed local differential basis branch
        dh_dt_basis = (
            self.alpha  * self.lap(h) +
            self.beta_x * self.dx(h) +
            self.beta_y * self.dy(h)
        )

        dh_dt = dh_dt_main + dh_dt_basis
        return dh_dt

class ODE_ConvGRU_Func(nn.Module):
    """
    ODE RHS wrapper for torchdiffeq.

    Supports two modes:
      - Controlled ODE (recommended for SR): uses a time-varying guidance U(t)
        built from the temporally-interpolated LR sequence.
            dh/dt = cell(h, U(t))
      - Autonomous ODE (your old behavior): uses constant x_input (e.g., zeros)
            dh/dt = cell(h, x_input)
    """
    def __init__(self, cell):
        super().__init__()
        self.cell = cell

        # Autonomous forcing (old)
        self.x_input = None          # [B,C,H,W]

        # Controlled forcing (new)
        self.U_seq = None            # [T_out,B,C,H,W]
        self.t0 = None
        self.t1 = None

    #@torch.no_grad()
    def set_control(self, U_seq: torch.Tensor, t_steps: torch.Tensor):
        """
        U_seq:   [T_out,B,C,H,W]  (C should match hidden dim)
        t_steps: [T_out] (monotonic)
        """
        self.U_seq = U_seq
        self.t0 = float(t_steps[0].item())
        self.t1 = float(t_steps[-1].item())

    def _U_at(self, t: torch.Tensor) -> torch.Tensor:
        U = self.U_seq
        if U is None:
            return None
    
        T = U.shape[0]
        tt = torch.clamp(t, min=self.t0, max=self.t1).to(device=U.device, dtype=U.dtype)
        s = (tt - self.t0) / (self.t1 - self.t0 + 1e-12) * (T - 1)
    
        i0 = torch.floor(s).clamp(0, T - 2).long()
        i1 = i0 + 1
        w = (s - i0.float()).view(1, 1, 1, 1)
    
        return (1 - w) * U[i0] + w * U[i1]

    def forward(self, t, h):
        # Controlled path if U_seq is set
        if self.U_seq is not None:
            u_t = self._U_at(t)
            return self.cell(h, u_t)

        # Fallback autonomous path (old)
        if self.x_input is None:
            self.x_input = torch.zeros_like(h)
        return self.cell(h, self.x_input)
        

class TemporalResidualRefiner(nn.Module):
    """
    对 [T,B,C,H,W] 在时间维做轻量残差修正
    """
    def __init__(self, channels, hidden=None, kernel_size=3):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd"
        hidden = hidden or max(16, channels * 4)
        pad = kernel_size // 2

        self.in_proj = nn.Conv1d(channels, hidden, kernel_size=1)
        self.tconv1 = nn.Conv1d(hidden, hidden, kernel_size=kernel_size, padding=pad)
        self.act = nn.GELU()
        self.tconv2 = nn.Conv1d(hidden, hidden, kernel_size=kernel_size, padding=pad)
        self.out_proj = nn.Conv1d(hidden, channels, kernel_size=1)

    def forward(self, x):
        """
        x: [T,B,C,H,W]
        """
        T, B, C, H, W = x.shape

        z = x.permute(1, 3, 4, 2, 0).contiguous()   # [B,H,W,C,T]
        z = z.view(B * H * W, C, T)                 # [BHW,C,T]

        z0 = z
        z = self.in_proj(z)
        z = self.act(self.tconv1(z))
        z = self.act(self.tconv2(z))
        z = self.out_proj(z)

        z = z + z0

        z = z.view(B, H, W, C, T)
        z = z.permute(4, 0, 3, 1, 2).contiguous()   # [T,B,C,H,W]
        return z
        
class temporal_sr(nn.Module):
    """
    方案 B:
    先线性插值，再做 temporal conv residual refinement
    输入:  [T, B, C, H, W]
    输出: [T*factor, B, C, H, W]
    """
    def __init__(self, t_upscale_factor, channels=2, hidden=32, kernel_size=5):
        super().__init__()
        self.t_upscale_factor = t_upscale_factor
        self.refiner = TemporalResidualRefiner(
            channels=channels,
            hidden=hidden,
            kernel_size=kernel_size
        )

    def forward(self, x):
        t, b, c, h, w = x.shape

        z = x.permute(1, 3, 4, 2, 0)                 # [B,H,W,C,T]
        z = z.contiguous().view(b * h * w, c, t)
        z = F.interpolate(
            z,
            scale_factor=self.t_upscale_factor,
            mode='linear',
            align_corners=True
        )
        z = z.contiguous().view(b, h, w, c, t * self.t_upscale_factor)
        z = z.permute(4, 0, 3, 1, 2).contiguous()   # [T',B,C,H,W]

        z = self.refiner(z)
        return z
        

class PhySR_ODE_ConvGRU(nn.Module):
    def __init__(self, n_feats, n_layers, upscale_factor,
                 shift_mean_paras_lr, shift_mean_paras_hr, custom_act=None):
        super().__init__()
        self.n_next_blocks, self.n_resblock = n_layers
        self.t_up_factor, self.s_up_factor = upscale_factor

        mean_lres, std_lres = shift_mean_paras_lr
        mean_hres, std_hres = shift_mean_paras_hr

        self.tsr = temporal_sr(self.t_up_factor)

        # Encoder
        self.feat_extract = nn.Conv2d(2, n_feats, kernel_size=3, padding=1, padding_mode='circular')
        self.h0_proj = nn.Conv2d(n_feats, n_feats, kernel_size=1)

        # ODE-ConvGRU (你原封不动保留)
        self.gru_cell = ConvNeXt_GRU_Cell(n_feats, custom_act=custom_act)
        self.ode_wrapper = ODE_ConvGRU_Func(self.gru_cell)
        
        def wn_circ_conv3x3(in_ch, out_ch):
            return weight_norm(nn.Conv2d(in_ch, out_ch, 3, padding=1, padding_mode='circular'))

        self.tail = WaveletSRHead(n_feats, out_ch=2, scale=self.s_up_factor, conv3x3_factory=wn_circ_conv3x3)
        self.skip = WaveletSRHead(2,      out_ch=2, scale=self.s_up_factor, conv3x3_factory=wn_circ_conv3x3)


        self.shift_in = ShiftMean(mean_lres, std_lres)
        self.shift_out = ShiftMean(mean_hres, std_hres)

    def forward(self, x, t_steps=None, odeint=None, method='rk4', use_control=True):
        """
        x: [T_in, B, 2, H, W]
        Returns: [T_out, B, 2, H*s_up, W*s_up]
        """
        T_in, B, C, H, W = x.shape
        x = self.shift_in(x, mode='sub')

        # 1) Temporal interpolation in LR space
        x_lr_up = self.tsr(x)                 # [T_out,B,2,H,W]
        T_out = x_lr_up.shape[0]

        # 2) time grid
        if t_steps is None:
            t_steps = torch.linspace(0, float(T_out - 1), T_out, device=x.device, dtype=x.dtype)
        else:
            t_steps = t_steps.to(device=x.device, dtype=x.dtype)
            if t_steps.numel() != T_out:
                raise ValueError(f"t_steps length {t_steps.numel()} must equal T_out {T_out}")

        # 3) Build guidance U_seq
        x_flat = x_lr_up.reshape(T_out * B, C, H, W)
        U_flat = self.feat_extract(x_flat)                 # [T_out*B,n_feats,H,W]
        U_seq = U_flat.view(T_out, B, -1, H, W)            # [T_out,B,n_feats,H,W]

        h = self.h0_proj(U_seq[0])
        hs = []
        dt = t_steps[1] - t_steps[0]
        for k in range(T_out):
            # dh = self.gru_cell(h, U_seq[k])
            # h = h + dt * dh
            k1 = self.gru_cell(h, U_seq[k])
            h_mid = h + 0.5 * dt * k1
            k2 = self.gru_cell(h_mid, U_seq[k])
            h = h + dt * k2

            hs.append(h)

        h_stack = torch.stack(hs, dim=0)                   # [T_out,B,n_feats,H,W]

        # ✅ 4) Vectorized spatial SR (把 T 并到 batch，一次性做完)
        h_flat = h_stack.reshape(T_out * B, -1, H, W)       # [T_out*B,n_feats,H,W]
        xlr_flat = x_lr_up.reshape(T_out * B, 2, H, W)      # [T_out*B,2,H,W]

        y_flat = self.tail(h_flat) + self.skip(xlr_flat)    # [T_out*B,2,H*s,W*s]
        y = y_flat.view(T_out, B, 2, H*self.s_up_factor, W*self.s_up_factor)

        y = self.shift_out(y, mode='add')
        return y



class Conv2dDerivative(nn.Module):
    def __init__(self, DerFilter, resol, kernel_size=3, name=''):
        super(Conv2dDerivative, self).__init__()

        self.resol = resol  # constant in the finite difference
        self.name = name
        self.input_channels = 1
        self.output_channels = 1
        self.kernel_size = kernel_size

        self.padding = int((kernel_size - 1) / 2)
        self.filter = nn.Conv2d(self.input_channels, self.output_channels, self.kernel_size, 
            1, padding=0, bias=False)

        # Fixed gradient operator
        self.filter.weight = nn.Parameter(torch.FloatTensor(DerFilter), requires_grad=False)  

    def forward(self, input):
        derivative = self.filter(input)
        return derivative / self.resol


class Conv1dDerivative(nn.Module):
    def __init__(self, DerFilter, resol, kernel_size=3, name=''):
        super(Conv1dDerivative, self).__init__()

        self.resol = resol  # $\delta$*constant in the finite difference
        self.name = name
        self.input_channels = 1
        self.output_channels = 1
        self.kernel_size = kernel_size

        self.padding = int((kernel_size - 1) / 2)
        self.filter = nn.Conv1d(self.input_channels, self.output_channels, self.kernel_size, 
            1, padding=0, bias=False)
        # Fixed gradient operator
        self.filter.weight = nn.Parameter(torch.FloatTensor(DerFilter), requires_grad=False)  

    def forward(self, input):
        derivative = self.filter(input)
        return derivative / self.resol


class GLossGenerator(nn.Module):
    def __init__(self, dt=0.002, dx=2.0/256, device='cuda',
                 eps=0.004, nu=10.0, gamma_re=10.0, gamma_im=15.0):
        super(GLossGenerator, self).__init__()

        self.eps = eps
        self.nu = nu
        self.gamma_re = gamma_re
        self.gamma_im = gamma_im

        self.laplace = Conv2dDerivative(
            DerFilter=lapl_op, resol=(dx**2), kernel_size=5, name='laplace_operator'
        ).to(device)

        self.dt = Conv1dDerivative(
            DerFilter=[[[-1/2, 0, 1/2]]], resol=dt, kernel_size=3, name='partial_t'
        ).to(device)

        self.fwd_dt = Conv1dDerivative(
            DerFilter=[[[-3/2, 2, -1/2]]], resol=dt, kernel_size=3, name='forward_partial_t'
        ).to(device)

        self.bwd_dt = Conv1dDerivative(
            DerFilter=[[[1/2, -2, 3/2]]], resol=dt, kernel_size=3, name='backward_partial_t'
        ).to(device)

    def _time_derivative(self, x):
        # x: [T,B,1,H,W] -> [T,B,1,H,W]
        T, B, C, H, W = x.shape
        z = x.permute(3, 4, 1, 2, 0).reshape(H * W * B, C, T)

        z_t_mid = self.dt(z)
        z_t0 = self.fwd_dt(z[:, :, 0:3])
        z_tn = self.bwd_dt(z[:, :, -3:])
        z_t = torch.cat((z_t0, z_t_mid, z_tn), dim=2)

        z_t = z_t.reshape(H, W, B, C, T).permute(4, 2, 3, 0, 1)
        return z_t

    def _lap(self, x):
        # x: [T,B,1,H,W] -> [T,B,1,H-4,W-4]
        T, B, C, H, W = x.shape
        z = x.reshape(T * B, C, H, W)
        z = self.laplace(z)
        z = z.reshape(T, B, C, H - 4, W - 4)
        return z

    def GetPhyLoss(self, output):
        """
        output: [T,B,2,H_pad,W_pad], already periodic padded outside
        return: res_u, res_v (both [T,B,1,Hc,Wc], Hc=H_pad-4)
        """
        u = output[:, :, 0:1, :, :]
        v = output[:, :, 1:2, :, :]

        lap_u = self._lap(u)
        lap_v = self._lap(v)

        # align to laplace valid area
        u_c = output[:, :, 0:1, 2:-2, 2:-2]
        v_c = output[:, :, 1:2, 2:-2, 2:-2]

        u_t = self._time_derivative(u_c)
        v_t = self._time_derivative(v_c)

        amp2 = u_c**2 + v_c**2

        # GL PDE residuals
        # u_t = eps*Δu + nu*u - amp2*(gamma_re*u - gamma_im*v)
        # v_t = eps*Δv + nu*v - amp2*(gamma_re*v + gamma_im*u)
        res_u = self.eps * lap_u + self.nu * u_c - amp2 * (self.gamma_re * u_c - self.gamma_im * v_c) - u_t
        res_v = self.eps * lap_v + self.nu * v_c - amp2 * (self.gamma_re * v_c + self.gamma_im * u_c) - v_t

        return res_u, res_v



def LossGenGL(output, truth, beta, loss_func=None, use_phy_loss=True, uv_beta=(1.0, 1.0)):
    """
    output, truth: [T,B,2,H,W]
    beta: global physics weight
    uv_beta: (w_u, w_v) residual weights
    """
    L1_loss = nn.L1Loss()
    MSE_loss = nn.MSELoss()

    data_loss = L1_loss(output, truth)

    if (not use_phy_loss) or (loss_func is None) or (beta <= 0.0):
        zero = torch.zeros(1, device=output.device, dtype=output.dtype)
        return data_loss, data_loss, zero, zero, zero

    # periodic padding (halo=2, symmetric)
    output_pad = torch.cat((output[:, :, :, :, -2:], output, output[:, :, :, :, :2]), dim=4)
    output_pad = torch.cat((output_pad[:, :, :, -2:, :], output_pad, output_pad[:, :, :, :2, :]), dim=3)

    res_u, res_v = loss_func.GetPhyLoss(output_pad)

    phy_u = MSE_loss(res_u, torch.zeros_like(res_u))
    phy_v = MSE_loss(res_v, torch.zeros_like(res_v))
    phy_loss = uv_beta[0] * phy_u + uv_beta[1] * phy_v

    total_loss = data_loss + beta * phy_loss
    return total_loss, data_loss, phy_loss, phy_u, phy_v


def train_gl(model, train_loader, val_loader, n_iters, lr, print_every,
             dt, dx, beta, device,
             phy_start_epoch=300, beta_ramp_epochs=300,
             eps=0.004, nu=10.0, gamma_re=10.0, gamma_im=15.0,
             uv_beta=(1.0, 1.0),
             best_ckpt_path=None,
             hist_save_path=None):

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_iters, eta_min=1e-7)

    loss_function = GLossGenerator(
        dt=dt, dx=dx, device=device,
        eps=eps, nu=nu, gamma_re=gamma_re, gamma_im=gamma_im
    )

    best_error = 1e9
    mse = nn.MSELoss()

    hist = {
        "train_loss": [],
        "train_data_loss": [],
        "train_phy_loss": [],
        "train_phy_u": [],
        "train_phy_v": [],
        "val_loss": [],
        "val_data_loss": [],
        "val_phy_loss": [],
        "val_phy_u": [],
        "val_phy_v": [],
        "val_error": [],
        "lr": [],
        "beta_epoch": [],
        "mode": []
    }

    for epoch in range(n_iters):
        if epoch < phy_start_epoch:
            use_phy_loss = False
            beta_epoch = 0.0
            mode_now = "warmup_data_only"
        else:
            use_phy_loss = True
            ramp_k = min(1.0, (epoch - phy_start_epoch + 1) / float(beta_ramp_epochs))
            beta_epoch = beta * ramp_k
            mode_now = "data_plus_gl_phy"

        model.train()
        tr_tot = tr_data = tr_phy = tr_u = tr_v = 0.0
        nb = 0

        for lres, hres in train_loader:
            lres = lres.to(device).transpose(0, 1)
            hres = hres.to(device).transpose(0, 1)

            optimizer.zero_grad()
            out = model(lres)

            total_loss, data_loss, phy_loss, phy_u, phy_v = LossGenGL(
                out, hres, beta=beta_epoch, loss_func=loss_function,
                use_phy_loss=use_phy_loss, uv_beta=uv_beta
            )

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

            tr_tot += total_loss.item()
            tr_data += data_loss.item()
            tr_phy += phy_loss.item()
            tr_u += phy_u.item()
            tr_v += phy_v.item()
            nb += 1

        scheduler.step()

        tr_tot_avg = tr_tot / nb
        tr_data_avg = tr_data / nb
        tr_phy_avg = tr_phy / nb
        tr_u_avg = tr_u / nb
        tr_v_avg = tr_v / nb

        model.eval()
        va_tot = va_data = va_phy = va_u = va_v = va_rel = 0.0
        nv = 0

        with torch.no_grad():
            for lres, hres in val_loader:
                lres = lres.to(device).transpose(0, 1)
                hres = hres.to(device).transpose(0, 1)
                out = model(lres)

                total_loss, data_loss, phy_loss, phy_u, phy_v = LossGenGL(
                    out, hres, beta=beta_epoch, loss_func=loss_function,
                    use_phy_loss=use_phy_loss, uv_beta=uv_beta
                )

                rel_err = torch.sqrt(mse(hres, out) / (mse(hres, torch.zeros_like(hres)) + 1e-12))

                va_tot += total_loss.item()
                va_data += data_loss.item()
                va_phy += phy_loss.item()
                va_u += phy_u.item()
                va_v += phy_v.item()
                va_rel += rel_err.item()
                nv += 1

        va_tot_avg = va_tot / nv
        va_data_avg = va_data / nv
        va_phy_avg = va_phy / nv
        va_u_avg = va_u / nv
        va_v_avg = va_v / nv
        va_rel_avg = va_rel / nv

        hist["train_loss"].append(tr_tot_avg)
        hist["train_data_loss"].append(tr_data_avg)
        hist["train_phy_loss"].append(tr_phy_avg)
        hist["train_phy_u"].append(tr_u_avg)
        hist["train_phy_v"].append(tr_v_avg)

        hist["val_loss"].append(va_tot_avg)
        hist["val_data_loss"].append(va_data_avg)
        hist["val_phy_loss"].append(va_phy_avg)
        hist["val_phy_u"].append(va_u_avg)
        hist["val_phy_v"].append(va_v_avg)
        hist["val_error"].append(va_rel_avg)

        hist["lr"].append(optimizer.param_groups[0]['lr'])
        hist["beta_epoch"].append(beta_epoch)
        hist["mode"].append(mode_now)

        if (epoch + 1) % print_every == 0 or epoch == 0:
            print(f"[Epoch {epoch+1}/{n_iters}] mode={mode_now}, beta_epoch={beta_epoch:.3e}, lr={optimizer.param_groups[0]['lr']:.2e}")
            print(f"train: total={tr_tot_avg:.4e}, data={tr_data_avg:.4e}, phy={tr_phy_avg:.4e}, phy_u={tr_u_avg:.4e}, phy_v={tr_v_avg:.4e}")
            print(f"val  : total={va_tot_avg:.4e}, data={va_data_avg:.4e}, phy={va_phy_avg:.4e}, phy_u={va_u_avg:.4e}, phy_v={va_v_avg:.4e}, rel={va_rel_avg:.4e}")

        if va_rel_avg < best_error:
            best_error = va_rel_avg

            if best_ckpt_path is not None:
                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_error": best_error,
                }, best_ckpt_path)

            print(f"New best checkpoint saved. best_rel_err={best_error:.6e}")

    hist["best_error"] = best_error

    if hist_save_path is not None:
        torch.save({
            "hist": hist
        }, hist_save_path)
        print(f"Full history saved to {hist_save_path}")

    return hist



# ==================== DATASET CLASS ====================
class GLDataset(Dataset):
    def __init__(self, data_dir, data_fname, ICs, random_influence=0.1, use_noisy_lr=False, obsn_tag="obsn00"):
        self.samples = []
        ri_tag = f"ri{int(random_influence * 1000):03d}"   # 0.1 -> ri100

        for ic in ICs:
            if use_noisy_lr:
                lres_filename = f'{data_fname}{ic}_{ri_tag}_{obsn_tag}_2x300x32x32.mat'
            else:
                lres_filename = f'{data_fname}{ic}_{ri_tag}_2x300x32x32.mat'
            hres_filename = f'{data_fname}{ic}_{ri_tag}_2x1200x256x256.mat'

            lres = scio.loadmat(os.path.join(data_dir, lres_filename))['uv']  # [2,300,32,32]
            hres = scio.loadmat(os.path.join(data_dir, hres_filename))['uv']  # [2,1200,256,256]

            lres = np.transpose(lres, (1, 0, 2, 3))  # [300,2,32,32]
            hres = np.transpose(hres, (1, 0, 2, 3))  # [1200,2,256,256]

            # 4 组样本，LR 25 帧 -> HR 100 帧
            l_starts = [100, 150, 200, 250]
            for ls in l_starts:
                hs = ls * 4
                self.samples.append((
                    torch.tensor(lres[ls:ls+25], dtype=torch.float32),
                    torch.tensor(hres[hs:hs+100], dtype=torch.float32)
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]
 
if __name__ == '__main__':
    data_fname = '2DGL_IC'
    RANDOM_INFLUENCE = 0.1

    ICs = np.arange(1, 11)
    
    test_ic =7  # 先固定 test IC
    
    # 剩余IC
    remaining_ICs = [ic for ic in ICs if ic != test_ic]
    
    # 随机选7个做train
    train_ICs = random.sample(remaining_ICs, 7)
    
    # 剩下的做val
    val_ICs = [ic for ic in remaining_ICs if ic not in train_ICs]
    
    # test集合
    test_ICs = [test_ic]
    
    print("train_ICs:", train_ICs)
    print("val_ICs:", val_ICs)
    print("test_ICs:", test_ICs)

    print("\nLoading datasets by IC split...")

    train_data = GLDataset(
        data_dir=data_dir,
        data_fname=data_fname,
        ICs=train_ICs,
        random_influence=RANDOM_INFLUENCE,
        use_noisy_lr=False
    )

    val_data = GLDataset(
        data_dir=data_dir,
        data_fname=data_fname,
        ICs=val_ICs,
        random_influence=RANDOM_INFLUENCE,
        use_noisy_lr=False
    )

    test_data = GLDataset(
        data_dir=data_dir,
        data_fname=data_fname,
        ICs=test_ICs,
        random_influence=RANDOM_INFLUENCE,
        use_noisy_lr=False
    )

    if len(train_data) == 0 or len(val_data) == 0 or len(test_data) == 0:
        raise ValueError("One of train/val/test datasets is empty! Please check your data files.")

    print(f"Train samples: {len(train_data)}")  # 7*4=28
    print(f"Val samples  : {len(val_data)}")    # 2*4=8
    print(f"Test samples : {len(test_data)}")   # 1*4=4

    # 动态 shape（从 train 集拿一个样本即可）
    sample_lres, sample_hres = train_data[0]
    Tlr, Clr, Hlr, Wlr = sample_lres.shape
    Thr, Chr, Hhr, Whr = sample_hres.shape
    print(f"One sample LR shape: {sample_lres.shape}")
    print(f"One sample HR shape: {sample_hres.shape}")

    # 仅用 train 集统计均值方差（推荐）
    train_lres = torch.stack([train_data[i][0] for i in range(len(train_data))])  # [Ntr,Tlr,2,Hlr,Wlr]
    train_hres = torch.stack([train_data[i][1] for i in range(len(train_data))])  # [Ntr,Thr,2,Hhr,Whr]

    mean_lres = train_lres.mean(dim=(0, 1, 3, 4))  # [2]
    std_lres  = train_lres.std (dim=(0, 1, 3, 4))  # [2]
    mean_hres = train_hres.mean(dim=(0, 1, 3, 4))  # [2]
    std_hres  = train_hres.std (dim=(0, 1, 3, 4))  # [2]

    print(f"mean_lres: {mean_lres}")
    print(f"std_lres : {std_lres}")
    print(f"mean_hres: {mean_hres}")
    print(f"std_hres : {std_hres}\n")

    # DataLoader
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=4, shuffle=True,  num_workers=0)
    val_loader   = torch.utils.data.DataLoader(val_data,   batch_size=4, shuffle=False, num_workers=0)
    test_loader  = torch.utils.data.DataLoader(test_data,  batch_size=4, shuffle=False, num_workers=0)

    
    ######################### Build model #############################
    print("\nBuilding model...")
    
    # Training parameters
    n_iters = 1000
    learning_rate = 5e-3
    print_every = 2
    dt = 0.0168
    dx = 2.0 / 256
    steps = 100
    effective_step = list(range(0, steps))

    # physics schedule (GL)
    beta = 5e-3                 
    phy_start_epoch = 300    
    beta_ramp_epochs = 300     
    
    print('Super-Resolution for 2D GL equation...')
    
    model = PhySR_ODE_ConvGRU(
        n_feats=64,
        n_layers=[6, 3],
        upscale_factor=[4, 8],   # GL: 32 -> 256
        shift_mean_paras_lr=[mean_lres, std_lres],
        shift_mean_paras_hr=[mean_hres, std_hres],
        custom_act=None
    ).to(device)


    def count_parameters(model):
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return total_params, trainable_params
    
    total, trainable = count_parameters(model)
    print(f'{"-"*30}')
    print(f'Total parameters: {total:,}')
    print(f'Trainable parameters: {trainable:,}')
    print(f'{"-"*30}')

    save_dir = str(model_dir)
    os.makedirs(save_dir, exist_ok=True)
    
    best_ckpt_path = os.path.join(save_dir, "Ginzburg-Landau_SLD_best_model.pth")
    hist_save_path = os.path.join(save_dir, "Ginzburg-Landau_SLD_history.pth")
    
    def count_parameters(model):
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return total_params, trainable_params
    
    total, trainable = count_parameters(model)
    print(f'{"-"*30}')
    print(f'Total parameters: {total:,}')
    print(f'Trainable parameters: {trainable:,}')
    print(f'{"-"*30}')
    
    # Training
    print("\nStarting training...")
    start = time.time()
    
    # 如果你沿用我前面给你的 train_gl（返回hist字典）
    hist = train_gl(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        n_iters=n_iters,
        lr=learning_rate,
        print_every=print_every,
        dt=dt,
        dx=dx,
        beta=beta,
        device=device,
    
        phy_start_epoch=300,
        beta_ramp_epochs=300,
    
        best_ckpt_path=best_ckpt_path,
        hist_save_path=hist_save_path,
    )
    # 兼容你原来的变量名
    train_loss_list = hist["train_loss"]
    val_loss_list = hist["val_loss"]
    val_error_list = hist["val_error"]
    
    end = time.time()
    print(f'Training time: {(end - start):.2f} seconds')
    print(f'Best val rel error: {hist["best_error"]:.6e}')
